<a href="https://colab.research.google.com/github/emilsar/NLP-TCGA/blob/main/notebooks/Fall2026/lab2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 — Finding Similar Reports

Chapter 1 of the book took six made-up sentences, turned each into a row of word counts,
measured how alike two rows are, and labelled a new report by copying the label of the
most similar known one. This lab does exactly the same thing to the 9,523 real reports
from Lab 1 — no new methods, just the real data.

Then it asks a harder question: when the computer says two reports are similar, *why*?
You will build a small tool that shows the most similar reports and the words they share,
and use it to find out what the method is really matching on. The answer is not what you
would hope.

Two kinds of puzzle:

- **Code puzzles** — fill in each `___` and run the cell. An `assert` line tells you
  immediately whether you got it right.
- **THINK puzzles** — no code. Read the question in the comments and write your answer on
  the `# ANSWER:` line, in a sentence or two. The book section named in the puzzle has
  what you need.

The book: [Chapter 1, From Words to Vectors](https://obooks.tech/nlp/words-to-vectors.html).

## Setup

In [ ]:
import pandas as pd
import numpy as np

BASE = "https://raw.githubusercontent.com/emilsar/NLP-TCGA/main/data"

# This is Lab 1 in four lines: read both files, cut out the barcode, look up each
# report's cancer type. Nothing new here.
reports = pd.read_csv(f"{BASE}/corpus/TCGA_Reports.csv")
labels = pd.read_csv(f"{BASE}/cancer_type/tcga_patient_to_cancer_type.csv")
reports["patient_id"] = reports["patient_filename"].apply(lambda x: x.split(".")[0])
reports["cancer_type"] = labels.set_index("patient_id").loc[reports["patient_id"], "cancer_type"].values

reports[["patient_id", "cancer_type"]].head()

In [ ]:
# The number to beat, from Lab 1: always guess the most common cancer type.
baseline = reports["cancer_type"].value_counts(normalize=True).max()
print("Majority-class baseline:", round(baseline, 3))

## Known reports and new reports

In Chapter 1 we had six reports whose diagnosis we knew, and one new report whose
diagnosis we wanted to predict. Here we make the same two piles from the real data: most
reports become the **known** reports, and the rest are set aside as **new** reports.

We do know the answers for the new reports too — that is the point. The method never
sees those answers; we keep them only to check its guesses afterwards.

In [ ]:
# PUZZLE 1 — set aside 20% of the reports as new ones. Keep each cancer type in the same
# proportion in both piles. The second blank is the name of the column that holds the
# answer.
from sklearn.model_selection import train_test_split

known, new = train_test_split(
    reports, test_size=___, stratify=reports[___], random_state=0
)

assert len(new) == 1905
print(len(known), "known reports,", len(new), "new reports")

In [ ]:
# PUZZLE 2 — THINK
# Why bother setting reports aside? Suppose we skipped that step and simply asked the
# method to label every known report, then checked how many it got right. Remember that
# the method labels a report by finding the most similar known report. Which known report
# would that be, every single time, and what would the score tell us?
# Chapter 1, "Finding the most similar report".
#
# ANSWER: ___

In [ ]:
# PUZZLE 3 — THINK
# Lab 1 found that the rarest cancer type has only 43 reports. What could go wrong with
# that type if we picked the new reports purely at random, without stratify?
#
# ANSWER: ___

## From words to vectors

Exactly as in Chapter 1, sections *Turning language into a matrix* and *Transforming a
new report*: `CountVectorizer` builds a vocabulary and counts how many times each
vocabulary word appears in each report.

In [ ]:
# PUZZLE 4 — two method names. The vocabulary must be built from the known reports
# only; the new reports are then counted using that same vocabulary.
# Chapter 1, "Transforming a new report".
from sklearn.feature_extraction.text import CountVectorizer

vec = CountVectorizer()
X_known = vec.___(known["text"])
X_new = vec.___(new["text"])

assert X_new.shape[1] == X_known.shape[1]
print("known:", X_known.shape, "  new:", X_new.shape)

In [ ]:
# PUZZLE 5 — THINK
# Why `transform` for the new reports, and not `fit_transform`? What would go wrong if we
# built a second vocabulary from the new reports?
# Chapter 1, "Transforming a new report".
#
# ANSWER: ___

The vocabulary has almost 39,000 words, so each report has become a row of almost 39,000
numbers. Almost all of them are zero: any one report uses only a few hundred of those
words. That is why `CountVectorizer` hands back a **sparse matrix**, which stores only the
non-zero entries (Appendix B, *A sparse matrix*).

In [ ]:
# PUZZLE 6 — what fraction of the cells are not zero? A sparse matrix knows how many
# non-zero entries it holds; Appendix B, "A sparse matrix", names the attribute.

nonzero_share = X_known.___ / (X_known.shape[0] * X_known.shape[1])

assert nonzero_share < 0.01
print(f"{nonzero_share:.2%} of the cells are not zero")

In [ ]:
# For comparison: storing every cell, zeros included, at 8 bytes per number.
full_size_gb = X_known.shape[0] * X_known.shape[1] * 8 / 1e9
print(f"Stored in full this table would take {full_size_gb:.1f} GB.")
print("So never call .toarray() on it - that is exactly what builds the full table.")

In [ ]:
# One report's row: its most frequent words. Appendix B, "The top-N idiom".
words = vec.get_feature_names_out()
row = X_new[0].toarray().ravel()      # fine for ONE row
top = row.argsort()[-12:][::-1]

print("Report:", new["patient_id"].iloc[0], " cancer type:", new["cancer_type"].iloc[0])
for j in top:
    print(f"  {words[j]:<15} {row[j]}")

In [ ]:
# PUZZLE 7 — THINK
# That row is the report's document vector. What does it keep from the report, and what
# does it throw away? Make up two short sentences with opposite clinical meanings that
# would get exactly the same row.
# Chapter 1, "What did we preserve?" and "What did we lose?".
#
# ANSWER: ___

In [ ]:
# PUZZLE 8 — THINK
# A new report contains the misspelling "carcinma", which never occurs in any known
# report. What happens to that word when the report is counted?
# Chapter 1, "Out-of-vocabulary words".
#
# ANSWER: ___

## The most similar report

Chapter 1, *Measuring similarity between reports*: **cosine similarity** compares two
rows of counts. It is 1 when two reports use words in exactly the same proportions and 0
when they share no words at all.

In [ ]:
# PUZZLE 9 — compare the first new report with every known report, then find the
# position of the largest similarity. You know .max(), which gives the largest value
# (Appendix B, "Functions over a whole array"). The method that gives its *position*
# instead is spelled the same way with "arg" in front.
from sklearn.metrics.pairwise import cosine_similarity

sims = cosine_similarity(X_new[0], X_known).ravel()
best = sims.___()

assert sims[best] == sims.max()
print("New report:          ", new["cancer_type"].iloc[0])
print("Most similar known:  ", known["cancer_type"].iloc[best], f"(similarity {sims[best]:.2f})")

In [ ]:
# PUZZLE 10 — THINK
# Lab 1 found that bladder reports run about ten times longer than melanoma reports.
# Why does that make cosine similarity a better choice here than the plain dot product?
# Chapter 1, "Why not simply use the dot product?".
#
# ANSWER: ___

## A nearest-neighbour classifier

Chapter 1, *A nearest-neighbor classifier*: give every new report the cancer type of its
single most similar known report. scikit-learn does the whole comparison in one go.

In [ ]:
# PUZZLE 11 — one neighbour, cosine similarity. Then the share of new reports labelled
# correctly: a column of True/False, averaged. (The first two blanks are a number and a
# string.)
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(n_neighbors=___, metric="___")
knn.fit(X_known, known["cancer_type"])

predicted = knn.predict(X_new)
accuracy = (predicted == new["cancer_type"]).___()

assert accuracy > 0.8
print("Nearest-neighbour accuracy:", round(accuracy, 3))
print("Baseline:                  ", round(baseline, 3))

In [ ]:
# PUZZLE 12 — THINK
# In your own words: what does the baseline measure, and what does the gap between it and
# the nearest-neighbour accuracy tell you?
# Chapter 1, "Why we start with six sentences instead" (the paragraph on the baseline).
#
# ANSWER: ___

## The find-similar tool

About 85% right is a strong number — but a number cannot tell you *why*. The tool below
takes one new report and shows its most similar known reports: how similar, what cancer
type, the start of the text, and **the words that contributed most to the match**.

A word's contribution is its count in one report times its count in the other: cosine
similarity is built from exactly those products (Chapter 1, *Measuring similarity between
reports*). So the words with the biggest products are the ones doing the matching.

In [ ]:
# PUZZLE 13 — the positions of the k largest similarities, largest first.
# Appendix B, "The top-N idiom": the last k positions of argsort, then reversed.

def find_similar(i, k=3):
    sims = cosine_similarity(X_new[i], X_known).ravel()
    top = sims.argsort()[___][::-1]

    print("NEW REPORT", new["patient_id"].iloc[i], "-", new["cancer_type"].iloc[i])
    print("   ", new["text"].iloc[i][:150], "...")
    for j in top:
        shared = X_new[i].multiply(X_known[j]).toarray().ravel()
        biggest = [words[w] for w in shared.argsort()[-8:][::-1]]
        print()
        print(f"  similarity {sims[j]:.2f}   {known['patient_id'].iloc[j]} - {known['cancer_type'].iloc[j]}")
        print("   ", known["text"].iloc[j][:150], "...")
        print("    top shared words:", ", ".join(biggest))

find_similar(0)

In [ ]:
# PUZZLE 14 — THINK
# Look at the "top shared words" lines above. How many of those words would a pathologist
# call medical? What does that tell you about what "similar" means to this method?
#
# ANSWER: ___

In [ ]:
# Try it on other reports - change the number. Here is one the classifier got wrong:
wrong = np.where(predicted != new["cancer_type"].values)[0]
print(len(wrong), "new reports were labelled wrongly. The first one:")
print()
find_similar(wrong[0])

Look through a few more, right and wrong. Do the matched reports read like the same
*disease*, or like the same *kind of form*?

## Where did the neighbour come from?

Every barcode has three parts. The middle one names the **tissue source site** — the
hospital or laboratory that sent the sample to TCGA:

    TCGA-UW-A7GN
         ^^
         tissue source site

In [ ]:
# PUZZLE 15 — cut the barcode at the dashes and keep the middle piece.
# Appendix A, "Splitting and joining".

reports["site"] = reports["patient_id"].apply(lambda b: b.split(___)[___])

assert reports["site"].iloc[0] == reports["patient_id"].iloc[0][5:7]
print(reports["site"].nunique(), "different site codes")

In [ ]:
# PUZZLE 16 — for each site code, how many different cancer types does it send?
# Appendix B, "Grouping". The blank is the method that counts *distinct* values:
# "number of unique", squashed into one word.

types_per_site = reports.groupby("site")["cancer_type"].___()

print("Most cancer types sent by any one site code:", types_per_site.max())

One. Every site code sends exactly one cancer type. (TCGA gives a hospital a separate
code for each cancer study it contributed to, so a code really means "this hospital's
reports for this one cancer".)

In [ ]:
# PUZZLE 17 — THINK
# Suppose a new report's nearest known report comes from the same site code. What do you
# know, for certain, about whether the prediction is right?
#
# ANSWER: ___

In [ ]:
# Which site did each new report's nearest neighbour come from?
known_site = reports.loc[known.index, "site"].values
new_site = reports.loc[new.index, "site"].values

nearest = knn.kneighbors(X_new, n_neighbors=1, return_distance=False)[:, 0]
neighbour_site = known_site[nearest]

In [ ]:
# PUZZLE 18 — what share of new reports have a nearest neighbour from their own site?
# Same idea as the accuracy in Puzzle 11: True/False, averaged.

same_site = (neighbour_site == new_site).___()

assert 0.7 < same_site < 0.9
print("Nearest neighbour from the same site:", round(same_site, 3))

In [ ]:
# Is that a lot? Compare with chance. If the neighbour were just a random known report of
# the right cancer type, how often would it come from the same site?
right = predicted == new["cancer_type"].values
print("Of the correct predictions, neighbour from the same site:", round((neighbour_site == new_site)[right].mean(), 3))

known_types = known["cancer_type"].values
chance = np.mean([(known_site[known_types == t] == s).mean()
                  for t, s in zip(new["cancer_type"], new_site)])
print("A random known report of the right type, same site:     ", round(chance, 3))

In [ ]:
# PUZZLE 19 — THINK
# About 92% against about 13%. When nearest neighbour gets a report right, what is it
# mostly matching on? Hint: reports from one hospital are written by the same pathologists,
# on the same template, with the same headings and stock phrases.
#
# ANSWER: ___

## Test it: reports from sites the method has never seen

If the method is mostly recognising a site's paperwork, it should struggle with a site it
has never met. We can check. Instead of picking new reports at random, pick whole **sites**
at random: every report from those sites becomes new, and none of them stays known.

In [ ]:
# Same as Puzzle 1, except that the split keeps each site's reports together.
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=0)
known_rows, new_rows = next(splitter.split(reports, groups=reports["site"]))
known2, new2 = reports.iloc[known_rows], reports.iloc[new_rows]

vec2 = CountVectorizer()
knn2 = KNeighborsClassifier(n_neighbors=1, metric="cosine")
knn2.fit(vec2.fit_transform(known2["text"]), known2["cancer_type"])
accuracy2 = (knn2.predict(vec2.transform(new2["text"])) == new2["cancer_type"]).mean()

print("New reports from sites it has seen:    ", round(accuracy, 3))
print("New reports from sites it has never seen:", round(accuracy2, 3))

In [ ]:
# PUZZLE 20 — THINK
# Chapter 1 warned that a model which learns "hospital B mostly sends kidney cases" has
# learned something true about the dataset and useless about pathology (Chapter 1,
# "Things a feature can measure"). A hospital that has never sent TCGA a report wants to
# use this method. Which of the two accuracies above should you quote them, and why?
#
# ANSWER: ___

## What you did

- Turned 9,523 real reports into word-count vectors, exactly as Chapter 1 did with six
  sentences — and saw why the table has to stay sparse.
- Kept a pile of new reports aside, so the classifier is graded on reports it has not
  seen.
- Labelled each new report by its most similar known report: about 85% right, against a
  baseline of 11%.
- Built a tool to look inside that number, and found that many of the shared words carry
  no medical meaning.
- Found that the method largely matches *where a report came from*, and that its accuracy
  falls by half on reports from hospitals it has never seen.

None of that came from a better model. It came from refusing to stop at the first number.
Two things you saw here come back later: the shared words that carry no meaning (Chapter 2
deals with them), and the question of how to split data so that a score means what you
think it means (Chapter 3).